In [ ]:
# ==============================================================================
# CSCI 544 Project: Hallucination Detection Baseline
# Method: N-Gram Subspace Features + MLP (Reference: Jerry Li, 2025)
# ==============================================================================



import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# ---------------------------------------------------------
# Step 1: Data Pipeline (HaluEval)
# ---------------------------------------------------------
print("1. Loading HaluEval QA dataset from Hugging Face...")
dataset = load_dataset("pminervini/HaluEval", "qa")
df = pd.DataFrame(dataset['data'])

print("2. Restructuring data for binary classification...")

df_factual = df[['question', 'right_answer']].copy()
df_factual.columns = ['prompt', 'response']
df_factual['label'] = 0


df_hallucinated = df[['question', 'hallucinated_answer']].copy()
df_hallucinated.columns = ['prompt', 'response']
df_hallucinated['label'] = 1


df_combined = pd.concat([df_factual, df_hallucinated]).sample(frac=1, random_state=42).reset_index(drop=True)


df_combined['text_to_analyze'] = df_combined['prompt'] + " [SEP] " + df_combined['response']


df_subset = df_combined.head(10000)

X_train, X_test, y_train, y_test = train_test_split(
    df_subset['text_to_analyze'], df_subset['label'], test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# Step 2: The Methodological Pipeline (N-Grams -> SVD -> MLP)
# ---------------------------------------------------------
print("\n3. Building the Jerry Li (2025) Extraction Pipeline...")


ngram_vectorizer = CountVectorizer(ngram_range=(1, 3), max_df=0.90, min_df=3)


svd_compressor = TruncatedSVD(n_components=100, random_state=42)


mlp_classifier = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500, # 500 epochs to ensure convergence
    random_state=42,
    early_stopping=True # Stops if validation score isn't improving to save time
)

# Bundle into a Scikit-Learn Pipeline
model_pipeline = Pipeline([
    ('ngrams', ngram_vectorizer),
    ('svd', svd_compressor),
    ('mlp', mlp_classifier)
])

# ---------------------------------------------------------
# Step 3: Training and Evaluation
# ---------------------------------------------------------
print("\n4. Training the SVD + MLP Pipeline (This may take a minute on CPU)...")
model_pipeline.fit(X_train, y_train)

print("\n5. Running Evaluation Metrics...")
# Predict classes and probabilities
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1] # Get probability of Class 1 (Hallucination)

# Output results exactly as outlined in your proposal
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=['Factual (0)', 'Hallucination (1)']))

print("=== CONTINUOUS UNCERTAINTY METRICS ===")
print(f"AUROC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"AUPRC: {average_precision_score(y_test, y_prob):.4f}")

1. Loading HaluEval QA dataset from Hugging Face...
2. Restructuring data for binary classification...

3. Building the Jerry Li (2025) Extraction Pipeline...

4. Training the SVD + MLP Pipeline (This may take a minute on CPU)...

5. Running Evaluation Metrics...

=== CLASSIFICATION REPORT ===
                   precision    recall  f1-score   support

      Factual (0)       0.75      0.86      0.80       984
Hallucination (1)       0.85      0.72      0.78      1016

         accuracy                           0.79      2000
        macro avg       0.80      0.79      0.79      2000
     weighted avg       0.80      0.79      0.79      2000

=== CONTINUOUS UNCERTAINTY METRICS ===
AUROC: 0.8542
AUPRC: 0.8794


In [ ]:
# ==============================================================================
# CSCI 544 Project: Hallucination Detection Baseline
# Method: N-Gram Subspace Features + MLP (Reference: Jerry Li, 2025)
# ==============================================================================



import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# ---------------------------------------------------------
# Step 1: Data Pipeline (HaluEval)
# ---------------------------------------------------------
print("1. Loading HaluEval QA dataset from Hugging Face...")
dataset = load_dataset("pminervini/HaluEval", "dialogue")
df = pd.DataFrame(dataset['data'])

print("2. Restructuring data for binary classification...")

# Create factual samples (Label 0)
df_factual = df[['dialogue_history', 'right_response']].copy()
df_factual.columns = ['prompt', 'response']
df_factual['label'] = 0

# Create hallucinated samples (Label 1)
df_hallucinated = df[['dialogue_history', 'hallucinated_response']].copy()
df_hallucinated.columns = ['prompt', 'response']
df_hallucinated['label'] = 1

df_combined = pd.concat([df_factual, df_hallucinated]).sample(frac=1, random_state=42).reset_index(drop=True)


df_combined['text_to_analyze'] = df_combined['prompt'] + " [SEP] " + df_combined['response']


df_subset = df_combined.head(10000)

X_train, X_test, y_train, y_test = train_test_split(
    df_subset['text_to_analyze'], df_subset['label'], test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# Step 2: The Methodological Pipeline (N-Grams -> SVD -> MLP)
# ---------------------------------------------------------
print("\n3. Building the Jerry Li (2025) Extraction Pipeline...")


ngram_vectorizer = CountVectorizer(ngram_range=(1, 3), max_df=0.90, min_df=3)


svd_compressor = TruncatedSVD(n_components=100, random_state=42)


mlp_classifier = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500, # 500 epochs to ensure convergence
    random_state=42,
    early_stopping=True # Stops if validation score isn't improving to save time
)

# Bundle into a Scikit-Learn Pipeline
model_pipeline = Pipeline([
    ('ngrams', ngram_vectorizer),
    ('svd', svd_compressor),
    ('mlp', mlp_classifier)
])

# ---------------------------------------------------------
# Step 3: Training and Evaluation
# ---------------------------------------------------------
print("\n4. Training the SVD + MLP Pipeline (This may take a minute on CPU)...")
model_pipeline.fit(X_train, y_train)

print("\n5. Running Evaluation Metrics...")
# Predict classes and probabilities
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1] # Get probability of Class 1 (Hallucination)

# Output results exactly as outlined in your proposal
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=['Factual (0)', 'Hallucination (1)']))

print("=== CONTINUOUS UNCERTAINTY METRICS ===")
print(f"AUROC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"AUPRC: {average_precision_score(y_test, y_prob):.4f}")

1. Loading HaluEval QA dataset from Hugging Face...


dialogue/data-00000-of-00001.parquet:   0%|          | 0.00/3.45M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

2. Restructuring data for binary classification...

3. Building the Jerry Li (2025) Extraction Pipeline...

4. Training the SVD + MLP Pipeline (This may take a minute on CPU)...

5. Running Evaluation Metrics...

=== CLASSIFICATION REPORT ===
                   precision    recall  f1-score   support

      Factual (0)       0.59      0.65      0.61       984
Hallucination (1)       0.62      0.56      0.59      1016

         accuracy                           0.60      2000
        macro avg       0.60      0.60      0.60      2000
     weighted avg       0.60      0.60      0.60      2000

=== CONTINUOUS UNCERTAINTY METRICS ===
AUROC: 0.6363
AUPRC: 0.6939


In [ ]:
# ==============================================================================
# CSCI 544 Project: Hallucination Detection Baseline
# Method: N-Gram Subspace Features + MLP (Reference: Jerry Li, 2025)
# ==============================================================================



import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# ---------------------------------------------------------
# Step 1: Data Pipeline (HaluEval)
# ---------------------------------------------------------
print("1. Loading HaluEval QA dataset from Hugging Face...")
dataset = load_dataset("pminervini/HaluEval", "summarization")
df = pd.DataFrame(dataset['data'])

print("2. Restructuring data for binary classification...")

# Create factual samples (Label 0)
df_factual = df[['document', 'right_summary']].copy()
df_factual.columns = ['prompt', 'response']
df_factual['label'] = 0

# Create hallucinated samples (Label 1)
df_hallucinated = df[['document', 'hallucinated_summary']].copy()
df_hallucinated.columns = ['prompt', 'response']
df_hallucinated['label'] = 1

df_combined = pd.concat([df_factual, df_hallucinated]).sample(frac=1, random_state=42).reset_index(drop=True)


df_combined['text_to_analyze'] = df_combined['prompt'] + " [SEP] " + df_combined['response']


df_subset = df_combined.head(10000)

X_train, X_test, y_train, y_test = train_test_split(
    df_subset['text_to_analyze'], df_subset['label'], test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# Step 2: The Methodological Pipeline (N-Grams -> SVD -> MLP)
# ---------------------------------------------------------
print("\n3. Building the Jerry Li (2025) Extraction Pipeline...")


ngram_vectorizer = CountVectorizer(ngram_range=(1, 3), max_df=0.90, min_df=3)


svd_compressor = TruncatedSVD(n_components=100, random_state=42)


mlp_classifier = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500, # 500 epochs to ensure convergence
    random_state=42,
    early_stopping=True # Stops if validation score isn't improving to save time
)

# Bundle into a Scikit-Learn Pipeline
model_pipeline = Pipeline([
    ('ngrams', ngram_vectorizer),
    ('svd', svd_compressor),
    ('mlp', mlp_classifier)
])

# ---------------------------------------------------------
# Step 3: Training and Evaluation
# ---------------------------------------------------------
print("\n4. Training the SVD + MLP Pipeline (This may take a minute on CPU)...")
model_pipeline.fit(X_train, y_train)

print("\n5. Running Evaluation Metrics...")
# Predict classes and probabilities
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1] # Get probability of Class 1 (Hallucination)

# Output results exactly as outlined in your proposal
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=['Factual (0)', 'Hallucination (1)']))

print("=== CONTINUOUS UNCERTAINTY METRICS ===")
print(f"AUROC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"AUPRC: {average_precision_score(y_test, y_prob):.4f}")

1. Loading HaluEval QA dataset from Hugging Face...


summarization/data-00000-of-00001.parque(…):   0%|          | 0.00/28.0M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

2. Restructuring data for binary classification...

3. Building the Jerry Li (2025) Extraction Pipeline...

4. Training the SVD + MLP Pipeline (This may take a minute on CPU)...

5. Running Evaluation Metrics...

=== CLASSIFICATION REPORT ===
                   precision    recall  f1-score   support

      Factual (0)       0.50      0.46      0.48       984
Hallucination (1)       0.51      0.54      0.53      1016

         accuracy                           0.50      2000
        macro avg       0.50      0.50      0.50      2000
     weighted avg       0.50      0.50      0.50      2000

=== CONTINUOUS UNCERTAINTY METRICS ===
AUROC: 0.4933
AUPRC: 0.4966
